# Employee Attrition Intelligence Platform (EAIP)
### Predicting Employee Attrition with Machine Learning & Generating HR Business Insights

**Author:** Senior Data Science Team
**Notebook:** `01_Employee_Attrition_Analysis.ipynb`
**Dataset:** IBM HR Analytics Employee Attrition Dataset

---
This notebook follows an end-to-end, production-grade machine learning workflow:

1. Data Loading
2. Data Cleaning & Preprocessing (sklearn `Pipeline` + `ColumnTransformer`)
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Train/Test Split
6. Class Imbalance Handling
7. Model Building (5 algorithms)
8. Hyperparameter Tuning
9. Model Evaluation
10. Feature Importance
11. SHAP Explainability
12. Model Persistence
13. HR Business Insights & Recommendations

> **Note:** This notebook requires `xgboost` and `shap` in addition to the standard data-science stack
> (`pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`). Install with:
> `pip install xgboost shap` if not already available.


## 0. Environment Setup & Imports

We import all libraries up front, following production notebook conventions. We also set
global plotting style and random seeds for full reproducibility.

In [ ]:
# ----------------------------------------------------------------------------
# Core libraries
# ----------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Visualization
# ----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------------------------
# Preprocessing & Modeling
# ----------------------------------------------------------------------------
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)

# XGBoost (industry-standard gradient boosting library)
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("xgboost not installed -> run `pip install xgboost` to enable this model.")

# SHAP (model explainability)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("shap not installed -> run `pip install shap` to enable explainability section.")

import joblib

# ----------------------------------------------------------------------------
# Global configuration
# ----------------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

PALETTE = ["#2E5266", "#6E8898", "#D64550", "#9DB4C0", "#5C8001"]

print("Libraries imported successfully.")


## Task 1 — Data Loading

**Explanation:** We load the raw dataset and perform an initial inspection: shape, schema,
summary statistics, missing values, duplicates, target distribution, and a count of numeric
vs categorical features. This step establishes a baseline understanding of data quality and
structure before any transformation.

In [ ]:
# Load dataset
df = pd.read_csv("employee_attrition.csv")

# Strip potential BOM / whitespace from column names
df.columns = df.columns.str.strip()

print(f"Dataset loaded successfully with shape: {df.shape}")
df.head(10)


In [ ]:
# Shape of the dataset
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")


In [ ]:
# Dataset information: dtypes, non-null counts
df.info()


In [ ]:
# Statistical summary of numeric columns
df.describe().T


In [ ]:
# Statistical summary of categorical columns
df.describe(include="object").T


In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_count", ascending=False)

if missing_df.empty:
    print("No missing values detected in the dataset.")
else:
    display(missing_df)


In [ ]:
# Duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")


In [ ]:
# Target variable distribution
target_counts = df["Attrition"].value_counts()
target_pct = df["Attrition"].value_counts(normalize=True) * 100

print("Target distribution (counts):")
print(target_counts)
print("\nTarget distribution (%):")
print(target_pct.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition Class Distribution")
ax.set_xlabel("Attrition")
ax.set_ylabel("Employee Count")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()


In [ ]:
# Count of numeric vs categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")


**Observation:** The dataset contains 1,470 employee records across 35 columns with no
missing values and no duplicate rows — a clean starting point. The target variable
`Attrition` is imbalanced, with roughly 84% "No" vs 16% "Yes", which we must address before
modeling (Task 6). Several columns (`EmployeeCount`, `StandardHours`, `Over18`) are constant
across all rows and carry no predictive signal; `EmployeeNumber` is a unique identifier. All
four will be dropped in Task 2.

**Business Insight:** An ~16% attrition rate, if left unmanaged, translates into significant
recruitment, onboarding, and lost-productivity costs annually. Even modest reductions in this
rate represent measurable HR cost savings — this is the business case for the model we are
about to build.

## Task 2 — Data Cleaning

**Explanation:** We drop identifier/constant columns that add no predictive value, encode the
target variable numerically, and build a `ColumnTransformer`-based preprocessing pipeline
(`OneHotEncoder` for categoricals, `StandardScaler` for numerics). Preprocessing is done via
`sklearn.pipeline.Pipeline` rather than manual transformations, ensuring no data leakage and a
fully reproducible, production-ready workflow.

In [ ]:
# Drop unnecessary columns (identifiers / constant-value columns)
cols_to_drop = ["EmployeeNumber", "EmployeeCount", "StandardHours", "Over18"]
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f"Dropped columns: {cols_to_drop}")
print(f"Shape after dropping columns: {df_clean.shape}")


In [ ]:
# Encode target variable: Yes -> 1, No -> 0
df_clean["Attrition"] = df_clean["Attrition"].map({"Yes": 1, "No": 0})

print("Target encoding complete.")
df_clean["Attrition"].value_counts()


In [ ]:
# Separate features and target
X = df_clean.drop(columns=["Attrition"])
y = df_clean["Attrition"]

# Identify feature types for the ColumnTransformer
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")


In [ ]:
# Build preprocessing pipeline using ColumnTransformer
# - Numeric features  -> StandardScaler
# - Categorical features -> OneHotEncoder (drop first to avoid multicollinearity)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
    ],
    remainder="drop"
)

print("ColumnTransformer-based preprocessor defined.")
preprocessor


**Observation:** After removing non-informative columns, we retain a clean feature matrix
of numeric and categorical predictors. The preprocessing pipeline is defined declaratively
and will be fit only on training data inside each model's `Pipeline` (Task 7) to prevent
data leakage.

**Business Insight:** Standardizing preprocessing into a single reusable pipeline object means
this same transformation logic can be deployed directly into an HR production system,
guaranteeing consistent treatment of new employee records at inference time.

## Task 3 — Exploratory Data Analysis

**Explanation:** We visually explore relationships between key HR variables and attrition to
surface patterns that will both guide feature engineering and directly inform HR strategy.
Each plot is followed by a business insight.

### Department vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(x="Department", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Department")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

dept_rate = df.groupby("Department")["Attrition"].apply(lambda x: (x == "Yes").mean() * 100)
print("Attrition rate by department (%):")
print(dept_rate.sort_values(ascending=False).round(2))


**Business Insight:** Departments with disproportionately higher attrition rates relative to their headcount should be prioritized for retention interventions and exit-interview deep dives.

### JobRole vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.countplot(y="JobRole", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax,
              order=df["JobRole"].value_counts().index)
ax.set_title("Attrition by Job Role")
plt.tight_layout()
plt.show()

role_rate = df.groupby("JobRole")["Attrition"].apply(lambda x: (x == "Yes").mean() * 100)
print("Attrition rate by job role (%):")
print(role_rate.sort_values(ascending=False).round(2))


**Business Insight:** Roles with the highest attrition rates (often entry-level, high-pressure, or front-line roles such as Sales Representative and Laboratory Technician) warrant targeted retention programs and role-specific career development paths.

### Gender vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x="Gender", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Gender")
plt.tight_layout()
plt.show()


**Business Insight:** If attrition rates differ meaningfully by gender, HR should investigate whether underlying drivers (compensation equity, career growth, work-life balance) differ systematically between groups.

### Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="Age", hue="Attrition", kde=True, palette=PALETTE[:2], ax=ax, multiple="stack")
ax.set_title("Age Distribution by Attrition")
plt.tight_layout()
plt.show()


**Business Insight:** Attrition concentrated in younger age bands signals a need for stronger early-career engagement, mentorship, and growth-path visibility for new hires.

### Monthly Income vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(x="Attrition", y="MonthlyIncome", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Monthly Income by Attrition")
plt.tight_layout()
plt.show()

print(df.groupby("Attrition")["MonthlyIncome"].median())


**Business Insight:** Lower median income among leavers suggests compensation plays a role, but should be examined alongside satisfaction and tenure metrics rather than treated as the sole driver.

### Business Travel vs Attrition

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x="BusinessTravel", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Business Travel Frequency")
plt.tight_layout()
plt.show()


**Business Insight:** Frequent travelers often show elevated attrition; HR could consider travel-frequency caps or additional compensation/support for high-travel roles.

### Job Satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x="JobSatisfaction", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Job Satisfaction (1=Low, 4=High)")
plt.tight_layout()
plt.show()


**Business Insight:** Low job satisfaction is a strong, directly actionable lever — regular satisfaction pulse surveys paired with manager coaching can catch at-risk employees early.

### Environment Satisfaction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x="EnvironmentSatisfaction", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Environment Satisfaction (1=Low, 4=High)")
plt.tight_layout()
plt.show()


**Business Insight:** Poor workplace environment satisfaction correlates with attrition; facilities, team dynamics, and management practices should be audited in low-scoring teams.

### Years at Company

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="YearsAtCompany", hue="Attrition", kde=True, palette=PALETTE[:2], ax=ax, multiple="stack")
ax.set_title("Years at Company by Attrition")
plt.tight_layout()
plt.show()


**Business Insight:** Attrition risk is typically highest in the first 1-3 years; onboarding quality and early career support are critical retention levers in this window.

### Years Since Last Promotion

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="YearsSinceLastPromotion", hue="Attrition", kde=True, palette=PALETTE[:2], ax=ax, multiple="stack")
ax.set_title("Years Since Last Promotion by Attrition")
plt.tight_layout()
plt.show()


**Business Insight:** Stalled career progression is a classic attrition driver — proactive promotion cycles and transparent career ladders can mitigate this risk.

### Distance From Home

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(x="Attrition", y="DistanceFromHome", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Distance From Home by Attrition")
plt.tight_layout()
plt.show()


**Business Insight:** Longer commutes are associated with higher attrition; flexible/hybrid work policies for employees living far from the office could improve retention.

### Education

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x="Education", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Education Level (1=Below College ... 5=Doctor)")
plt.tight_layout()
plt.show()


**Business Insight:** Education level alone is rarely a strong standalone driver but helps contextualize career-growth expectations across employee segments.

### Marital Status

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(x="MaritalStatus", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by Marital Status")
plt.tight_layout()
plt.show()


**Business Insight:** Single employees often show higher attrition than married peers, likely reflecting fewer location/financial constraints — relevant for tailoring retention messaging.

### OverTime

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(x="OverTime", hue="Attrition", data=df, palette=PALETTE[:2], ax=ax)
ax.set_title("Attrition by OverTime Status")
plt.tight_layout()
plt.show()

overtime_rate = df.groupby("OverTime")["Attrition"].apply(lambda x: (x == "Yes").mean() * 100)
print("Attrition rate by OverTime (%):")
print(overtime_rate.round(2))


**Business Insight:** OverTime is typically one of the single strongest attrition predictors in this dataset. Workload management and overtime policy reform should be a top HR priority.

### Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.4, ax=ax)
ax.set_title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()


**Business Insight:** The heatmap surfaces multicollinear clusters (e.g., YearsAtCompany, YearsInCurrentRole, YearsWithCurrManager) — useful context when interpreting feature importance later, since correlated features can dilute each other's individual importance scores.

## Task 4 — Feature Engineering

**Explanation:** We create a small set of domain-meaningful derived features that compress
useful signal from existing columns. Each feature is included only if it is conceptually
meaningful for attrition prediction.

In [ ]:
df_fe = df_clean.copy()

# IncomePerYear: monthly income annualized
df_fe["IncomePerYear"] = df_fe["MonthlyIncome"] * 12

# YearsPerPromotion: tenure relative to promotion cadence (avoid divide-by-zero)
df_fe["YearsPerPromotion"] = df_fe["YearsAtCompany"] / (df_fe["YearsSinceLastPromotion"] + 1)

# ExperienceRatio: proportion of total working years spent at the current company
df_fe["ExperienceRatio"] = df_fe["YearsAtCompany"] / (df_fe["TotalWorkingYears"] + 1)

# AgeGroup: categorical bucketing of age
df_fe["AgeGroup"] = pd.cut(
    df_fe["Age"], bins=[17, 25, 35, 45, 60],
    labels=["18-25", "26-35", "36-45", "46-60"]
).astype(str)

# TenureGroup: categorical bucketing of company tenure
df_fe["TenureGroup"] = pd.cut(
    df_fe["YearsAtCompany"], bins=[-1, 2, 5, 10, 40],
    labels=["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
).astype(str)

print("Engineered features added: IncomePerYear, YearsPerPromotion, ExperienceRatio, AgeGroup, TenureGroup")
df_fe[["IncomePerYear", "YearsPerPromotion", "ExperienceRatio", "AgeGroup", "TenureGroup"]].head()


**Observation:** These engineered features translate raw tenure and compensation figures
into ratios and buckets that are often more directly interpretable by tree-based models and
HR stakeholders alike.

**Business Insight:** `YearsPerPromotion` and `TenureGroup` in particular give HR a quick,
explainable lens for identifying employees whose career progression has stalled relative to
their tenure — a common precursor to voluntary attrition.

## Task 5 — Train/Test Split

**Explanation:** We split the data 80/20 using a fixed `random_state=42` and stratify on the
target to preserve the original class ratio in both sets.

In [ ]:
X = df_fe.drop(columns=["Attrition"])
y = df_fe["Attrition"]

# Recompute feature type lists (now includes engineered features)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
    ],
    remainder="drop"
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True).round(3)}")


## Task 6 — Handling Class Imbalance

**Explanation:** With ~84/16 class split, naive models will be biased toward the majority
class. We address this primarily via `class_weight='balanced'` inside each estimator, which
re-weights the loss function without altering the dataset itself (no synthetic data risk).

**Alternative:** `SMOTE` (Synthetic Minority Oversampling Technique) from the `imbalanced-learn`
library is a common alternative that generates synthetic minority-class samples. It can be
plugged into the pipeline as an additional step (`imblearn.pipeline.Pipeline` with
`SMOTE(random_state=42)`) if class-weighting alone proves insufficient. We use
`class_weight='balanced'` here for simplicity and to avoid synthetic data leakage risk
during cross-validation.

In [ ]:
# class_weight='balanced' will be passed directly into each estimator in Task 7.
# Example (commented) of the SMOTE alternative:
#
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline
#
# smote_pipeline = ImbPipeline(steps=[
#     ("preprocessor", preprocessor),
#     ("smote", SMOTE(random_state=RANDOM_STATE)),
#     ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
# ])

print("Class imbalance strategy: class_weight='balanced' (SMOTE noted as alternative above).")


## Task 7 — Model Building

**Explanation:** We train five classification models, each wrapped in its own `Pipeline`
(preprocessor + classifier) to guarantee consistent, leak-free preprocessing.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
}

if XGBOOST_AVAILABLE:
    # XGBoost handles imbalance via scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    models["XGBoost"] = XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )

pipelines = {
    name: Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    for name, model in models.items()
}

trained_pipelines = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    print(f"Trained: {name}")


## Task 8 — Hyperparameter Tuning

**Explanation:** We tune Random Forest, Gradient Boosting, and XGBoost using
`RandomizedSearchCV` with stratified 5-fold cross-validation, optimizing for ROC AUC
(a robust metric under class imbalance).

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_distributions = {
    "Random Forest": {
        "classifier__n_estimators": [200, 300, 400, 500],
        "classifier__max_depth": [4, 6, 8, 10, None],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4],
    },
    "Gradient Boosting": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__max_depth": [2, 3, 4, 5],
        "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    },
}

if XGBOOST_AVAILABLE:
    param_distributions["XGBoost"] = {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__max_depth": [3, 4, 5, 6],
        "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    }

tuned_pipelines = {}
for name, param_dist in param_distributions.items():
    print(f"Tuning {name} ...")
    search = RandomizedSearchCV(
        estimator=pipelines[name],
        param_distributions=param_dist,
        n_iter=20,
        scoring="roc_auc",
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    tuned_pipelines[name] = search.best_estimator_
    trained_pipelines[name] = search.best_estimator_  # replace with tuned version
    print(f"  Best ROC AUC (CV): {search.best_score_:.4f}")
    print(f"  Best params: {search.best_params_}\n")


## Task 9 — Model Evaluation

**Explanation:** We evaluate every trained pipeline on the held-out test set across the full
standard classification metric suite, then assemble a single comparison dataframe.

In [ ]:
def evaluate_model(name, pipe, X_test, y_test):
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba),
    }
    return metrics, y_pred, y_proba

results = []
predictions = {}

for name, pipe in trained_pipelines.items():
    metrics, y_pred, y_proba = evaluate_model(name, pipe, X_test, y_test)
    results.append(metrics)
    predictions[name] = {"y_pred": y_pred, "y_proba": y_proba}

comparison_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
comparison_df


In [ ]:
# Classification reports for every model
for name in trained_pipelines:
    print(f"\n===== {name} =====")
    print(classification_report(y_test, predictions[name]["y_pred"]))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, len(trained_pipelines), figsize=(5 * len(trained_pipelines), 4))
if len(trained_pipelines) == 1:
    axes = [axes]

for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# Cross-validation scores (ROC AUC) for each model on training data
cv_results = {}
for name, pipe in trained_pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name}: CV ROC AUC = {scores.mean():.4f} (+/- {scores.std():.4f})")


In [ ]:
# ROC Curves — all models on one plot
fig, ax = plt.subplots(figsize=(8, 7))
for name, preds in predictions.items():
    fpr, tpr, _ = roc_curve(y_test, preds["y_proba"])
    auc = roc_auc_score(y_test, preds["y_proba"])
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random Baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# Precision-Recall Curves — all models on one plot
fig, ax = plt.subplots(figsize=(8, 7))
for name, preds in predictions.items():
    precision, recall, _ = precision_recall_curve(y_test, preds["y_proba"])
    ax.plot(recall, precision, label=name)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve Comparison")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()


**Observation:** The comparison dataframe ranks models by ROC AUC, which is the most
informative metric here given class imbalance. Tree-based ensembles (Random Forest, Gradient
Boosting, XGBoost) typically outperform plain Logistic Regression on recall for the minority
("Attrition = Yes") class, which matters most for an HR early-warning use case.

**Business Insight:** Precision/Recall trade-offs translate directly into HR resourcing
decisions: higher recall means catching more true at-risk employees (at the cost of some false
positives requiring unnecessary check-ins), while higher precision means fewer wasted retention
interventions. HR should pick the model/threshold combination that matches their intervention
capacity.

## Task 10 — Feature Importance

**Explanation:** We extract and visualize the top 10 most influential features from
Random Forest, XGBoost, and Logistic Regression to understand what drives each model's
predictions.

In [ ]:
def get_feature_names(preprocessor):
    """Recover human-readable feature names after ColumnTransformer."""
    num_names = preprocessor.transformers_[0][2]
    cat_encoder = preprocessor.transformers_[1][1]
    cat_names = cat_encoder.get_feature_names_out(preprocessor.transformers_[1][2])
    return list(num_names) + list(cat_names)

feature_names = get_feature_names(trained_pipelines["Random Forest"].named_steps["preprocessor"])
print(f"Total transformed features: {len(feature_names)}")


In [ ]:
# Random Forest feature importance
rf_model = trained_pipelines["Random Forest"].named_steps["classifier"]
rf_importance = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
rf_importance.head(10).sort_values().plot(kind="barh", color=PALETTE[0], ax=ax)
ax.set_title("Top 10 Feature Importances — Random Forest")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost feature importance
if XGBOOST_AVAILABLE and "XGBoost" in trained_pipelines:
    xgb_model = trained_pipelines["XGBoost"].named_steps["classifier"]
    xgb_importance = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    xgb_importance.head(10).sort_values().plot(kind="barh", color=PALETTE[2], ax=ax)
    ax.set_title("Top 10 Feature Importances — XGBoost")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("XGBoost not available in this environment.")


In [ ]:
# Logistic Regression coefficients (signed, interpretable direction of effect)
lr_model = trained_pipelines["Logistic Regression"].named_steps["classifier"]
lr_coef = pd.Series(lr_model.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
top10_lr = lr_coef.head(10).sort_values()
colors = [PALETTE[2] if v > 0 else PALETTE[0] for v in top10_lr.values]
top10_lr.plot(kind="barh", color=colors, ax=ax)
ax.set_title("Top 10 Logistic Regression Coefficients (red = increases attrition risk)")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()


**Observation:** Across models, `OverTime`, `MonthlyIncome`, `Age`, `TotalWorkingYears`,
and satisfaction-related features consistently rank among the top predictors, though tree
ensembles and the linear model may rank engineered features differently depending on how they
capture non-linear interactions.

**Business Insight:** Convergence of multiple models on the same top drivers (especially
OverTime and compensation/tenure features) gives HR high confidence that these are genuine,
actionable levers — not modeling artifacts.

## Task 11 — SHAP Explainability

**Explanation:** SHAP (SHapley Additive exPlanations) values provide a theoretically grounded,
model-agnostic way to explain both global feature behavior and individual predictions. We use
the best-performing tree-based model for SHAP analysis (tree explainers are fast and exact for
ensemble models).

In [ ]:
if SHAP_AVAILABLE:
    # Use the top model from the comparison table for explainability
    best_model_name = comparison_df.iloc[0]["Model"]
    best_pipeline = trained_pipelines[best_model_name]

    # Transform test data through the preprocessor for SHAP
    X_test_transformed = best_pipeline.named_steps["preprocessor"].transform(X_test)
    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()
    X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names)

    explainer = shap.TreeExplainer(best_pipeline.named_steps["classifier"])
    shap_values = explainer.shap_values(X_test_transformed_df)

    print(f"SHAP explainer built for: {best_model_name}")
else:
    print("SHAP not available in this environment — skipping explainability section.")


In [ ]:
# SHAP Summary Plot (global feature impact)
if SHAP_AVAILABLE:
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test_transformed_df, show=False)
    plt.title("SHAP Summary Plot")
    plt.tight_layout()
    plt.show()


In [ ]:
# SHAP Waterfall Plot — explain a single employee's prediction
if SHAP_AVAILABLE:
    sample_idx = 0  # first employee in the test set

    explanation = shap.Explanation(
        values=shap_values[sample_idx] if not isinstance(shap_values, list) else shap_values[1][sample_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray))
                    else explainer.expected_value[1],
        data=X_test_transformed_df.iloc[sample_idx],
        feature_names=feature_names
    )

    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(explanation, show=False)
    plt.title(f"SHAP Waterfall — Employee #{sample_idx} (Test Set)")
    plt.tight_layout()
    plt.show()

    predicted_prob = best_pipeline.predict_proba(X_test.iloc[[sample_idx]])[0, 1]
    actual_label = y_test.iloc[sample_idx]
    print(f"Predicted attrition probability: {predicted_prob:.2%}")
    print(f"Actual label: {'Attrition' if actual_label == 1 else 'No Attrition'}")


**Observation:** The SHAP summary plot ranks features by their global average impact on
model output while also showing the direction of effect (e.g., high OverTime pushes
predictions toward attrition). The waterfall plot decomposes one specific employee's
prediction into individual feature contributions.

**Business Insight:** SHAP enables HR Business Partners to have individualized, evidence-based
retention conversations — e.g., "this employee's risk is driven primarily by stalled promotion
and overtime hours," not a generic risk score.

## Task 12 — Model Persistence

**Explanation:** We persist the best-performing trained pipeline (model + fitted
preprocessor bundled together) and the standalone preprocessor object using `joblib`, ready
for deployment.

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]

print(f"Best model selected for deployment: {best_model_name}")
print(comparison_df.iloc[0])

# Save the full pipeline (preprocessor + classifier) as the production model
joblib.dump(best_pipeline, "best_model.pkl")

# Save the fitted preprocessor separately for any standalone use
joblib.dump(best_pipeline.named_steps["preprocessor"], "preprocessor.pkl")

print("\nSaved: best_model.pkl, preprocessor.pkl")


## Task 13 — HR Business Insights & Recommendations

**Explanation:** We synthesize EDA findings, feature importance results, and SHAP analysis
into direct answers to the HR questions the business cares about.

In [ ]:
# Recap: department and role attrition rates computed in Task 3
print("Attrition rate by Department (%):")
print(df.groupby("Department")["Attrition"].apply(lambda x: (x == "Yes").mean() * 100).sort_values(ascending=False).round(2))

print("\nAttrition rate by Job Role (%):")
print(df.groupby("JobRole")["Attrition"].apply(lambda x: (x == "Yes").mean() * 100).sort_values(ascending=False).round(2))

print("\nTop 10 features driving attrition (Random Forest importance):")
print(rf_importance.head(10))


### Key Findings

**Which department has the highest attrition?**
Based on the department-level attrition rates above, Sales typically shows the highest
attrition rate among the three departments, followed by Human Resources, with Research &
Development the most stable — though exact ranking should be read off the printed output
above for this specific run.

**Which role has the highest attrition?**
Front-line, high-pressure, and lower-tenure roles (commonly Sales Representative and
Laboratory Technician in this dataset) tend to show the highest attrition rates.

**Top 10 factors affecting attrition:**
Consistently across Random Forest, XGBoost, and Logistic Regression: OverTime, MonthlyIncome,
Age, TotalWorkingYears, YearsAtCompany / YearsInCurrentRole, JobSatisfaction,
EnvironmentSatisfaction, DistanceFromHome, StockOptionLevel, and WorkLifeBalance — see the
ranked importance output above for this run's exact ordering.

**Is salary alone responsible?**
No. While `MonthlyIncome` is a consistently important feature, satisfaction scores, overtime
status, and career progression metrics (promotion recency, role tenure) are equally or more
influential in many models — attrition is multi-causal, not purely compensation-driven.

**What policies should HR introduce?**
Overtime workload caps, structured promotion review cycles, manager training focused on job
satisfaction, flexible/hybrid arrangements for long-commute employees, and targeted retention
bonuses for high-risk, high-value roles.

**Who should HR target?**
Employees flagged by the model with high predicted attrition probability AND high business
value/criticality — particularly those in their first 1–3 years, working frequent overtime,
with no recent promotion.

### 10 Actionable Recommendations

1. Implement mandatory overtime caps and monitor hours for early-tenure employees.
2. Establish a transparent, time-bound promotion review cycle (e.g., every 18–24 months).
3. Launch quarterly job-satisfaction and environment-satisfaction pulse surveys with manager follow-up.
4. Create a structured 1–3 year onboarding and mentorship program for new hires.
5. Offer flexible/hybrid work options for employees with long commutes (high `DistanceFromHome`).
6. Conduct a compensation equity review for roles/departments with above-average attrition.
7. Build manager scorecards that include team satisfaction and attrition-risk metrics.
8. Deploy the trained model as a monthly HR dashboard flagging high-risk employees for proactive 1:1s.
9. Expand stock option / long-term incentive eligibility to reduce early attrition in critical roles.
10. Run exit-interview root-cause analysis specifically for the highest-attrition department and role identified above, feeding findings back into policy design.


## Summary

This notebook delivered a complete, production-style ML workflow for employee attrition:
clean data ingestion, leak-free preprocessing pipelines, rich EDA, meaningful feature
engineering, five trained and three hyperparameter-tuned models, a full evaluation suite,
feature importance and SHAP-based explainability, persisted model artifacts
(`best_model.pkl`, `preprocessor.pkl`), and a concrete, HR-actionable insights and
recommendations section.

**Next steps for production deployment:** wrap `best_model.pkl` in a scoring API or batch
job, schedule monthly re-scoring of the active workforce, and route high-risk flags into the
HR Business Partner workflow alongside SHAP-based individual explanations.
